# AI PDF Knowledge Assistant using RAG

**Portfolio project | Python · PyPDF · Sentence Transformers · FAISS · Gradio**

## Overview
This notebook extracts text from a PDF, splits it into chunks, creates semantic embeddings, retrieves relevant passages using FAISS, evaluates retrieval with a small test set, and provides a Gradio interface that displays evidence and source page numbers.

**Current scope:** this version demonstrates the retrieval pipeline. It displays retrieved evidence rather than generating an LLM answer. Gemini integration can be added once API access/model availability is confirmed.

**Run instructions:** Run cells from top to bottom. Upload a text-based PDF when prompted. Do not upload confidential documents or API keys.


## 1. Install dependencies

In [ ]:
%pip -q install pypdf sentence-transformers faiss-cpu gradio


## 2. Upload a PDF and extract text

In [ ]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload a PDF file.")
pdf_path = next(iter(uploaded))
reader = PdfReader(pdf_path)
pages = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""
    if text.strip():
        pages.append({"page": page_number, "text": text})

print("Total PDF pages:", len(reader.pages))
print("Pages with extractable text:", len(pages))
if not pages:
    raise ValueError("No text extracted. Try a text-based PDF; scanned PDFs may require OCR.")
print("\nPreview:\n", pages[0]["text"][:700])


## 3. Split the document into overlapping chunks

In [ ]:
def create_chunks(pages, chunk_size=500, overlap=100):
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("Use chunk_size > overlap >= 0.")
    chunks = []
    for page in pages:
        text = page["text"]
        start = 0
        while start < len(text):
            end = min(start + chunk_size, len(text))
            chunk_text = text[start:end].strip()
            if chunk_text:
                chunks.append({"text": chunk_text, "page": page["page"]})
            if end == len(text):
                break
            start = end - overlap
    return chunks

chunks = create_chunks(pages)
if not chunks:
    raise ValueError("No chunks were created.")
print("Chunks created:", len(chunks))
print("First chunk:\n", chunks[0]["text"][:500])


## 4. Generate semantic embeddings

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = embedding_model.encode(
    chunk_texts, normalize_embeddings=True, convert_to_numpy=True
).astype(np.float32)

print("Embedding matrix shape:", embeddings.shape)


## 5. Build the FAISS vector index

In [ ]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("FAISS index created. Vectors indexed:", index.ntotal)


## 6. Retrieve relevant passages

In [ ]:
def retrieve(question, top_k=4):
    if not question or not question.strip():
        return []
    if index.ntotal == 0:
        return []
    query_vector = embedding_model.encode(
        [question], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)
    k = min(max(1, int(top_k)), index.ntotal)
    scores, ids = index.search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx >= 0:
            results.append({
                "text": chunks[int(idx)]["text"],
                "page": chunks[int(idx)]["page"],
                "score": float(score)
            })
    return results

question = "How many annual leave days do employees receive?"
for rank, result in enumerate(retrieve(question, top_k=3), start=1):
    print(f"\nResult {rank} | PDF page {result['page']} | similarity {result['score']:.3f}")
    print(result["text"][:500])


## 7. Basic retrieval evaluation

In [ ]:
# These expected phrases are for the fictional practice handbook used in this project.
# Update them if you upload a different PDF.
test_cases = [
    {"question": "How many annual leave days do employees receive?", "expected": ["20 days", "annual leave"]},
    {"question": "What are the standard working hours?", "expected": ["9:30 AM", "6:00 PM"]},
    {"question": "What security practices are required?", "expected": ["strong passwords", "multi-factor authentication"]},
    {"question": "When should critical customer issues be escalated?", "expected": ["support manager"]},
]

def evaluate_retrieval(test_cases, top_k=3):
    passed = 0
    report = []
    for case in test_cases:
        results = retrieve(case["question"], top_k=top_k)
        retrieved_text = " ".join(item["text"] for item in results).lower()
        matched = [phrase for phrase in case["expected"] if phrase.lower() in retrieved_text]
        success = len(matched) == len(case["expected"])
        passed += int(success)
        report.append({"question": case["question"], "expected": case["expected"], "matched": matched, "passed": success})
    score = passed / len(test_cases) if test_cases else 0.0
    print(f"Phrase-check pass rate: {passed}/{len(test_cases)} ({score:.1%})")
    for item in report:
        print("\nQuestion:", item["question"], "| Passed:", item["passed"])
        print("Matched phrases:", item["matched"])
    return report, score

evaluation_report, evaluation_score = evaluate_retrieval(test_cases, top_k=3)


### Evaluation caveat
This is a small phrase-based sanity check, not a standardized RAG benchmark. It can produce false negatives if wording differs and does not evaluate answer faithfulness. Review retrieved passages manually and update expected phrases/pages for your chosen PDF.


## 8. Gradio interface: show retrieved evidence

In [ ]:
import gradio as gr

def chat_with_pdf(question):
    if not question or not question.strip():
        return "Please enter a question."
    results = retrieve(question, top_k=3)
    if not results:
        return "No passages were retrieved. Check that the PDF contains extractable text."
    output = [
        "## Retrieved evidence",
        "_This version displays search results; it does not generate an LLM answer._"
    ]
    for rank, item in enumerate(results, start=1):
        output.append(f"### Source {rank} — PDF page {item['page']}")
        output.append(f"Similarity score: {item['score']:.3f}")
        output.append(item["text"])
    return "\n\n".join(output)

demo = gr.Interface(
    fn=chat_with_pdf,
    inputs=gr.Textbox(label="Ask a question about your PDF", placeholder="How many annual leave days do employees receive?"),
    outputs=gr.Markdown(label="Retrieved evidence"),
    title="AI PDF Knowledge Assistant",
    description="Retrieve relevant passages from the uploaded PDF and view source page references.",
    examples=[
        ["How many annual leave days do employees receive?"],
        ["What are the standard working hours?"],
        ["What security practices are required?"]
    ]
)

demo.launch(share=True, debug=True)


## 9. Results, limitations, and next steps

### Implemented
- PDF text extraction with page metadata
- Overlapping character-based chunking
- Sentence Transformer embeddings
- FAISS similarity search
- Retrieved passages with page references
- Basic phrase-based retrieval checks
- Gradio interface

### Limitations
- Scanned PDFs may require OCR.
- Character-based chunking can split sentences or topics.
- Similarity scores do not prove factual relevance.
- The current UI displays retrieved evidence, not an LLM-generated answer.
- Evaluation uses a small hand-written test set and should not be presented as a robust benchmark.

### Future improvements
1. Add an LLM with a verified, available API model and evidence-grounded prompts.
2. Test unsupported questions and calibrate a relevance threshold on a validation set.
3. Compare chunk sizes and evaluate retrieval with Recall@k or MRR.
4. Add file validation, error handling, and a persistent vector store.
5. Add automated tests and deploy the app.

**GitHub safety:** Never commit API keys, confidential PDFs, or private data.
